# A3-M1 · 04 tiktoken
OpenAI 모델의 토크나이저. 처음 실행할 때 인코딩 파일을 한 번 내려받습니다.

## 10-1. 인코딩 고르기 — get_encoding / encoding_for_model

In [1]:
import tiktoken
o200k = tiktoken.get_encoding("o200k_base")          # GPT-4o·4.1 계열
cl100k = tiktoken.get_encoding("cl100k_base")        # GPT-4·3.5, text-embedding-3
print(o200k.name, o200k.n_vocab, "|", cl100k.name, cl100k.n_vocab)
for m in ["gpt-4o", "gpt-4o-mini", "gpt-4", "text-embedding-3-small"]:
    print(f"{m:<24} → {tiktoken.encoding_for_model(m).name}")

o200k_base 200019 | cl100k_base 100277
gpt-4o                   → o200k_base
gpt-4o-mini              → o200k_base
gpt-4                    → cl100k_base
text-embedding-3-small   → cl100k_base


## 10-2. encode · decode · 토큰 하나가 몇 바이트인가

In [2]:
s = "배송 언제 와요?"
ids = o200k.encode(s)
print("ids   :", ids)
print("decode:", o200k.decode(ids))
for i in ids:
    b = o200k.decode_single_token_bytes(i)
    print(f"{i:>7} {str(b):<40} {len(b)}바이트 → {b.decode('utf-8', errors='replace')!r}")

ids   : [185040, 171771, 103238, 7952, 30]
decode: 배송 언제 와요?
 185040 b'\xeb\xb0\xb0\xec\x86\xa1'              6바이트 → '배송'
 171771 b' \xec\x96\xb8\xec\xa0\x9c'             7바이트 → ' 언제'
 103238 b' \xec\x99\x80'                         4바이트 → ' 와'
   7952 b'\xec\x9a\x94'                          3바이트 → '요'
     30 b'?'                                     1바이트 → '?'


## 10-3. 같은 뜻, 다른 토큰 수 — 언어와 인코딩

In [3]:
pairs = [("배송은 언제 오나요?", "When will my order arrive?"),
         ("환불은 3영업일 이내에 처리됩니다.", "Refunds are processed within 3 business days."),
         ("냉장고 같은 대형 가전은 설치 기사가 방문합니다.", "Large appliances are installed by a technician.")]
print(f"{'문장':<40} {'글자':>4} {'cl100k':>7} {'o200k':>6}")
for ko, en in pairs:
    for s in (ko, en):
        print(f"{s:<40} {len(s):>4} {len(cl100k.encode(s)):>7} {len(o200k.encode(s)):>6}")

문장                                         글자  cl100k  o200k
배송은 언제 오나요?                                11      12      6
When will my order arrive?                 26       6      6
환불은 3영업일 이내에 처리됩니다.                        19      18     14
Refunds are processed within 3 business days.   45      10     10
냉장고 같은 대형 가전은 설치 기사가 방문합니다.                27      22     16
Large appliances are installed by a technician.   47       8      8


## 10-4. 특수 토큰과 오류

In [4]:
text = "주문 내역 <|endoftext|> 다음 문서"
try:
    o200k.encode(text)
except ValueError as e:
    print("ValueError:", str(e).splitlines()[0][:80], "...")
print("글자로 취급:", o200k.encode(text, disallowed_special=()))
print("특수로 허용:", o200k.encode(text, allowed_special={"<|endoftext|>"}))
print("특수 토큰 목록:", o200k.special_tokens_set)

ValueError: Encountered text corresponding to disallowed special token '<|endoftext|>'. ...
글자로 취급: [9273, 11397, 21566, 21925, 464, 91, 419, 1440, 919, 91, 29, 68527, 26387, 4865]
특수로 허용: [9273, 11397, 21566, 21925, 220, 199999, 68527, 26387, 4865]
특수 토큰 목록: {'<|endofprompt|>', '<|endoftext|>'}


## 11-1. 대화 메시지 토큰 수와 비용 어림

In [5]:
def count_chat_tokens(messages, enc=o200k, per_message=3, reply_prime=3):
    """메시지마다 역할·구분 토큰 몇 개가 더 붙는다는 OpenAI 안내를 따른 어림값"""
    total = 0
    for m in messages:
        total += per_message
        for v in m.values():
            total += len(enc.encode(v))
    return total + reply_prime

msgs = [{"role": "system", "content": "쇼핑몰 상담원. 근거만 사용해 답하라."},
        {"role": "user", "content": "[근거] 3만 원 이상 무료 배송입니다.\n[질문] 배송비 얼마예요?"}]
n_in = count_chat_tokens(msgs)
PRICE_IN_PER_1M = 0.40          # 예시 단가(USD/100만 토큰). 실제 값은 공식 가격표에서 확인해 바꾼다
print("입력 토큰:", n_in, f"| 1만 번 호출 비용 ≈ ${n_in * 10000 / 1e6 * PRICE_IN_PER_1M:.2f}")

입력 토큰: 49 | 1만 번 호출 비용 ≈ $0.20


## 11-2. 토큰 기준으로 자르기와 나누기

In [6]:
def truncate_tokens(text, max_tokens, enc=o200k):
    ids = enc.encode(text)
    return enc.decode(ids[:max_tokens]), len(ids)

def chunk_tokens(text, size=20, overlap=5, enc=o200k):
    ids = enc.encode(text)
    step = size - overlap
    return [enc.decode(ids[i:i + size]) for i in range(0, max(len(ids) - overlap, 1), step)]

doc = ("냉장고 같은 대형 가전은 설치 기사가 직접 방문합니다. 방문 일정은 문자로 안내됩니다. "
       "제주와 도서산간은 3~5일 더 걸립니다. 설치 후 포장재는 기사님이 수거합니다.")
short, n = truncate_tokens(doc, 15)
print(f"전체 {n}토큰 → 15토큰:", short)
for i, c in enumerate(chunk_tokens(doc)):
    print(f"조각 {i} ({len(o200k.encode(c))}토큰): {c}")

전체 54토큰 → 15토큰: 냉장고 같은 대형 가전은 설치 기사가 직접 방문
조각 0 (20토큰): 냉장고 같은 대형 가전은 설치 기사가 직접 방문합니다. 방문 일정은
조각 1 (20토큰): 합니다. 방문 일정은 문자로 안내됩니다. 제주와 도서산간은 3~
조각 2 (20토큰): 간은 3~5일 더 걸립니다. 설치 후 포장재는 기사님이
조각 3 (9토큰): 재는 기사님이 수거합니다.


## 11-3. 문장 경계를 지키며 토큰 수로 나누기

In [7]:
from kiwipiepy import Kiwi
kiwi = Kiwi()

def chunk_by_sentence(text, max_tokens=25, enc=o200k):
    chunks, cur, cur_n = [], [], 0
    for sent in (s.text for s in kiwi.split_into_sents(text)):
        n = len(enc.encode(sent))
        if cur and cur_n + n > max_tokens:          # 이 문장을 넣으면 넘친다 → 지금까지를 한 조각으로
            chunks.append(" ".join(cur)); cur, cur_n = [], 0
        cur.append(sent); cur_n += n
    if cur:
        chunks.append(" ".join(cur))
    return chunks

for i, c in enumerate(chunk_by_sentence(doc)):
    print(f"조각 {i} ({len(o200k.encode(c))}토큰): {c}")

조각 0 (17토큰): 냉장고 같은 대형 가전은 설치 기사가 직접 방문합니다.
조각 1 (9토큰): 방문 일정은 문자로 안내됩니다.
조각 2 (17토큰): 제주와 도서산간은 3~5일 더 걸립니다.
조각 3 (14토큰): 설치 후 포장재는 기사님이 수거합니다.
